In [ ]:
uv add toyaikit



In [1]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [2]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": ["query"]
    }
}

In [3]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": ["filename", "title", "description", "content"]
    }
}

In [4]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search

2) in the second interation, analyze the results from the previous search
   and perform 2 more searches

3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [5]:
from toyaikit.tools import Tools
agent_tools = Tools()


In [6]:
agent_tools.add_tool(search,search_tool)

In [7]:
agent_tools.add_tool(add_entry,add_entry_tool)

In [8]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'title': {'type': 'string',
     'description': 'The title of the documentation entry'},
    'description': {'type': 'string',
     'description': 'A short description summarizing the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'title', 'description', 'content']}}]

In [9]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

In [10]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    # chat_interface=,
    llm_client=llm_client
)

In [11]:
result = agent.loop('how can I create evidently dahsbord?')

In [24]:
print(result.last_message)

Here's a synthesized guide on how to create a dashboard, based on the gathered information:

### Steps to Create a Dashboard

1. **Connect to the Platform:**
   - First, ensure that you are connected to the platform and have created a project where you will store your reports.

2. **Adding Tabs:**
   - **To add a Tab:**
     - Enter "Edit" mode on the Dashboard (located at the top right corner).
     - Click the plus sign labeled "add Tab" on the left.
     - You can create a custom Tab by selecting "empty" and entering a desired name.
   - You also have the option to use pre-built tabs, which are available for different kinds of data evaluations.

3. **Adding Panels:**
   - You can add panels through either the user interface or Python API.
   - **Using the UI:**
     - In "Edit" mode, click "Add Panel" next to the Tab where you want to place it.
     - Follow the prompts to configure the panel, including choosing values to display, setting parameters like title, type, and size, and f

In [12]:
result.cost.input_cost


Decimal('0.0025722')

In [13]:
result.cost.output_cost

Decimal('0.0003756')

In [22]:
from toyaikit.chat.runners import DisplayingRunnerCallback
from toyaikit.chat.interface import IPythonChatInterface

In [24]:
chat_interface = IPythonChatInterface()

In [25]:
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [26]:
result = agent.loop(
    'how can I create evidently dahsbard?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [28]:
result2= agent.loop('show me the code',callback=runner_callback,previous_messages=result.all_messages)

-> Response received


In [34]:
agent= OpenAIResponsesRunner(tools=agent_tools,developer_prompt=instructions,chat_interface=chat_interface,llm_client=llm_client)


In [ ]:
result = agent.run()
stop